In [1]:
import os
import pandas as pd
import numpy as np

os.chdir('/Users/jacksonsharpe/QSS20_Final_Project_Sharpe/data')
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/jacksonsharpe/QSS20_Final_Project_Sharpe/data


In [7]:
def load_ess(filepath='ESSQSS20.csv'):
    """
    Load ESS cumulative file, select relevant columns, clean missing values,
    filter to native-born respondents, build attitude index, and map rounds to years.
    
    ESS encodes missing/refused responses as values >10 on 0-10 scales.
    We recode these to NaN so they don't pollute averages.
    
    brncntr == 1 means respondent was born in the survey country (native-born).
    We keep only native-born to focus on host population attitudes.
    
    We use Rounds 1-10 (2002-2020) only because DEMIG policy data ends in 2020.
    Round 11 (2023) is excluded since no matching policy data exists.
    """
    cols = [
        'cntry', 'essround', 'imbgeco', 'imueclt', 'imwbcnt',
        'brncntr', 'eisced', 'eduyrs', 'agea', 'gndr',
        'uempla', 'lrscale', 'stfeco'
    ]
    ess = pd.read_csv(filepath, usecols=cols)
    
    # Recode attitude variables — values above 10 are refusal/don't know codes
    for col in ['imbgeco', 'imueclt', 'imwbcnt']:
        ess[col] = ess[col].where(ess[col] <= 10)
    
    # Filter to native-born only
    ess = ess[ess['brncntr'] == 1].copy()
    
    # Build pro-immigration attitude index as mean of 3 questions
    # Higher = more pro-immigration (0=most anti, 10=most pro)
    ess['immig_attitude'] = ess[['imbgeco', 'imueclt', 'imwbcnt']].mean(axis=1)
    ess = ess.dropna(subset=['immig_attitude'])
    
    # Clean other variables — recode out-of-range values to NaN
    ess['eisced'] = ess['eisced'].where(ess['eisced'] <= 7)
    ess['lrscale'] = ess['lrscale'].where(ess['lrscale'] <= 10)
    ess['agea'] = ess['agea'].where(ess['agea'] <= 120)
    ess['eduyrs'] = ess['eduyrs'].where(ess['eduyrs'] <= 30)
    ess['stfeco'] = ess['stfeco'].where(ess['stfeco'] <= 10)
    
    # Map ESS round numbers to fieldwork years
    # Rounds 1-10 only (2002-2020) — Round 11 excluded due to no DEMIG coverage
    round_to_year = {
        1:2002, 2:2004, 3:2006, 4:2008, 5:2010,
        6:2012, 7:2014, 8:2016, 9:2018, 10:2020
    }
    ess['year'] = ess['essround'].map(round_to_year)
    
    # Drop Round 11 rows (year will be NaN since it's not in round_to_year)
    ess = ess.dropna(subset=['year'])
    ess['year'] = ess['year'].astype(int)
    
    # Binary indicator: economically dissatisfied = stfeco <= 4
    ess['econ_dissatisfied'] = (ess['stfeco'] <= 4).astype(int)
    
    return ess


def load_policy(filepath='migration_policy.xls'):
    """
    Load DEMIG policy database.
    Each row is one policy change. We keep only rows coded as
    'more restrictive' or 'less restrictive' and assign numeric scores.
    
    restrict_score: +1 = more restrictive, -1 = less restrictive
    """
    df = pd.read_csv(filepath, sep='\t')
    
    # Normalize case and strip whitespace for consistent filtering
    df['Restrictiveness'] = df['Restrictiveness'].str.strip().str.lower()
    df = df[df['Restrictiveness'].isin(['more restrictive', 'less restrictive'])].copy()
    
    # Assign numeric score: +1 restrictive, -1 liberalizing
    df['restrict_score'] = df['Restrictiveness'].map({
        'more restrictive': 1,
        'less restrictive': -1
    })
    
    # Convert Year to numeric — raw file stores it as string
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(int)
    
    return df


def make_country_crosswalk():
    """
    Maps full country names (used in DEMIG) to 2-letter ISO codes (used in ESS).
    Needed to merge the two datasets on a common country identifier.
    """
    return {
        'Austria':'AT', 'Belgium':'BE', 'Bulgaria':'BG', 'Croatia':'HR',
        'Cyprus':'CY', 'Czech Republic':'CZ', 'Denmark':'DK', 'Estonia':'EE',
        'Finland':'FI', 'France':'FR', 'Germany':'DE', 'Greece':'GR',
        'Hungary':'HU', 'Iceland':'IS', 'Ireland':'IE', 'Italy':'IT',
        'Latvia':'LV', 'Lithuania':'LT', 'Luxembourg':'LU', 'Malta':'MT',
        'Netherlands':'NL', 'Norway':'NO', 'Poland':'PL', 'Portugal':'PT',
        'Romania':'RO', 'Slovakia':'SK', 'Slovenia':'SI', 'Spain':'ES',
        'Sweden':'SE', 'Switzerland':'CH', 'United Kingdom':'GB'
    }


def build_policy_panel(policy, crosswalk):
    """
    For each country and each ESS wave, compute net restrictiveness score
    using only policy changes in the 2-year window before that ESS wave.
    
    net_restrict_score = (# restrictive actions) - (# liberalizing actions)
    Positive = net restrictive period, Negative = net liberalizing period
    
    Windows based on official ESS fieldwork periods.
    Rounds 1-10 only (2002-2020) to match DEMIG coverage.
    """
    ess_windows = {
        2002: (2000, 2002), 2004: (2002, 2004), 2006: (2004, 2006),
        2008: (2006, 2008), 2010: (2008, 2010), 2012: (2010, 2012),
        2014: (2012, 2014), 2016: (2014, 2016), 2018: (2016, 2018),
        2020: (2018, 2020)
    }
    
    policy['cntry'] = policy['Country'].map(crosswalk)
    
    rows = []
    for country in policy['cntry'].dropna().unique():
        cp = policy[policy['cntry'] == country]
        for year, (start, end) in ess_windows.items():
            subset = cp[(cp['Year'] >= start) & (cp['Year'] <= end)]
            n_restrict = (subset['restrict_score'] == 1).sum()
            n_liberal = (subset['restrict_score'] == -1).sum()
            rows.append({
                'cntry': country,
                'year': year,
                'net_restrict_score': int(n_restrict) - int(n_liberal),
                'n_restrict': int(n_restrict),
                'n_liberal': int(n_liberal),
                'n_policy_actions': len(subset)
            })
    
    return pd.DataFrame(rows)

In [8]:
print("Loading ESS data...")
ess = load_ess()

print(f"ESS after cleaning: {ess.shape}")
print(f"Native-born respondents: {len(ess):,}")
print(f"Countries: {ess['cntry'].nunique()}")
print(f"ESS rounds: {sorted(ess['essround'].unique())}")
print(f"\nMissing values after cleaning:")
print(ess[['immig_attitude', 'stfeco', 'eisced', 'agea', 'eduyrs']].isna().sum())
print(f"\nEcon dissatisfied split:")
print(ess['econ_dissatisfied'].value_counts())
print(f"\nMean attitude by ESS round:")
print(ess.groupby('essround')['immig_attitude'].mean().round(2))
ess.head()

Loading ESS data...
ESS after cleaning: (436194, 16)
Native-born respondents: 436,194
Countries: 39
ESS rounds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Missing values after cleaning:
immig_attitude       0
stfeco            8265
eisced            2487
agea              2539
eduyrs            6130
dtype: int64

Econ dissatisfied split:
econ_dissatisfied
0    232961
1    203233
Name: count, dtype: int64

Mean attitude by ESS round:
essround
1     5.05
2     4.86
3     4.97
4     4.88
5     4.75
6     5.02
7     5.09
8     4.98
9     5.11
10    5.27
Name: immig_attitude, dtype: float64


,essround,cntry,lrscale,stfeco,imbgeco,imueclt,imwbcnt,brncntr,gndr,agea,eduyrs,eisced,uempla,immig_attitude,year,econ_dissatisfied
0,1,AT,6.0,7.0,4.0,9.0,7.0,1,1,54.0,11.0,0.0,0,6.666667,2002,0
1,1,AT,6.0,0.0,10.0,10.0,5.0,1,1,50.0,14.0,0.0,0,8.333333,2002,1
2,1,AT,5.0,7.0,7.0,5.0,5.0,1,2,63.0,9.0,0.0,0,5.666667,2002,0
3,1,AT,5.0,6.0,10.0,10.0,10.0,1,1,44.0,18.0,0.0,0,10.000000,2002,0
5,1,AT,NaN,0.0,5.0,7.0,5.0,1,2,63.0,11.0,0.0,0,5.666667,2002,1


In [9]:
print("Loading DEMIG policy data...")
policy = load_policy()
crosswalk = make_country_crosswalk()

print(f"DEMIG after filtering to restrictive/liberalizing: {len(policy):,} rows")

print("\nBuilding policy panel...")
policy_panel = build_policy_panel(policy, crosswalk)

print(f"Policy panel shape: {policy_panel.shape}")
print(f"Missing net_restrict_score: {policy_panel['net_restrict_score'].isna().sum()}")
print(f"\nSample policy panel (Austria):")
print(policy_panel[policy_panel['cntry'] == 'AT'].to_string(index=False))

Loading DEMIG policy data...
DEMIG after filtering to restrictive/liberalizing: 4,927 rows

Building policy panel...
Policy panel shape: (310, 6)
Missing net_restrict_score: 0

Sample policy panel (Austria):
cntry  year  net_restrict_score  n_restrict  n_liberal  n_policy_actions
   AT  2002                   0           6          6                12
   AT  2004                   2           9          7                16
   AT  2006                   5          11          6                17
   AT  2008                   0           6          6                12
   AT  2010                   4           6          2                 8
   AT  2012                   2           9          7                16
   AT  2014                 -12           6         18                24
   AT  2016                   3          25         22                47
   AT  2018                   7          29         22                51
   AT  2020                  -4          13         17        

In [10]:
country_year_ess = (
    ess.groupby(['cntry', 'year'])['immig_attitude']
    .agg(mean_attitude='mean', n_respondents='count')
    .reset_index()
)

print(f"ESS country-year observations: {len(country_year_ess)}")
print(country_year_ess.head(10))

ESS country-year observations: 259
  cntry  year  mean_attitude  n_respondents
0    AL  2012       5.830940           1184
1    AT  2002       5.252310           2020
2    AT  2004       4.777859           2049
3    AT  2006       4.680505           2204
4    AT  2014       4.503378           1579
5    AT  2016       4.363291           1797
6    AT  2018       4.696123           2218
7    AT  2020       4.785714           1750
8    BE  2002       4.838557           1723
9    BE  2004       4.759843           1617


In [11]:
print(f"Before merge — ESS rows: {len(country_year_ess)}, Policy rows: {len(policy_panel)}")

merged = pd.merge(country_year_ess, policy_panel, on=['cntry', 'year'], how='inner')

print(f"After merge: {len(merged)} rows")
print(f"Countries in merged dataset: {merged['cntry'].nunique()}")
print(f"Years covered: {sorted(merged['year'].unique())}")
print(f"\nMissing values in merged dataset:")
print(merged.isna().sum())

merged.to_csv('merged_attitudes_policy.csv', index=False)
print(f"\nSaved to merged_attitudes_policy.csv")
merged.head(10)

Before merge — ESS rows: 259, Policy rows: 310
After merge: 233 rows
Countries in merged dataset: 30
Years covered: [np.int64(2002), np.int64(2004), np.int64(2006), np.int64(2008), np.int64(2010), np.int64(2012), np.int64(2014), np.int64(2016), np.int64(2018), np.int64(2020)]

Missing values in merged dataset:
cntry                 0
year                  0
mean_attitude         0
n_respondents         0
net_restrict_score    0
n_restrict            0
n_liberal             0
n_policy_actions      0
dtype: int64

Saved to merged_attitudes_policy.csv


,cntry,year,mean_attitude,n_respondents,net_restrict_score,n_restrict,n_liberal,n_policy_actions
0,AT,2002,5.252310,2020,0,6,6,12
1,AT,2004,4.777859,2049,2,9,7,16
2,AT,2006,4.680505,2204,5,11,6,17
3,AT,2014,4.503378,1579,-12,6,18,24
4,AT,2016,4.363291,1797,3,25,22,47
5,AT,2018,4.696123,2218,7,29,22,51
6,AT,2020,4.785714,1750,-4,13,17,30
7,BE,2002,4.838557,1723,-2,0,2,2
8,BE,2004,4.759843,1617,2,7,5,12
9,BE,2006,4.988030,1643,5,11,6,17
